# SGSMA 2026 - Synchrophasor Anomaly Detection

In [1]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 1 — Imports & Global Constants
# ═══════════════════════════════════════════════════════════════════════════
import os, warnings, time, json
import numpy as np
import pandas as pd
from scipy import stats as sp_stats
from numpy.lib.stride_tricks import sliding_window_view

from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    classification_report, confusion_matrix,
    precision_recall_curve
)
from sklearn.model_selection import TimeSeriesSplit

import xgboost as xgb
import lightgbm as lgb
import joblib

warnings.filterwarnings('ignore')
np.random.seed(42)

PMU_FILES = {
    'Bus2':  'Bus2_Competition_Data_nanmask.csv',
    'Bus5':  'Bus5_Competition_Data_nanmask.csv',
    'Bus6':  'Bus6_Competition_Data_nanmask.csv',
    'Bus10': 'Bus10_Competition_Data_nanmask.csv',
    'Bus19': 'Bus19_Competition_Data_nanmask.csv',
    'Bus22': 'Bus22_Competition_Data_nanmask.csv',
    'Bus29': 'Bus29_Competition_Data_nanmask.csv',
    'Bus39': 'Bus39_Competition_Data_nanmask.csv',
}

PMU_MEASURE_COLS = [
    'va_mag','va_ang','vb_mag','vb_ang','vc_mag','vc_ang',
    'ia_mag','ia_ang','ib_mag','ib_ang','ic_mag','ic_ang',
    'freq','rocof',
]

EVENT_NAMES = {
    0:'Normal', 1:'Fault', 2:'Line Outage',
    3:'Gen Change', 4:'Load Change', 5:'Missing Data',
    6:'Missing+Physical', 7:'Bad Data', 8:'Unknown',
}

# Deterministic label → bus mapping from competition Table 3
LABEL_TO_BUS = {
    0: 'N/A (normal)',
    1: 'Bus 39',
    2: 'Bus 24-Bus 23 (line)',
    3: 'Bus 2',
    4: 'Bus 7',
    5: 'Bus 29 (PMU comm. failure)',
    6: 'Bus 29 (missing) + Bus 2 (gen change)',
    7: 'Unknown (bad data)',
    8: 'Unknown',
}
LABEL_TO_BUS_TOP3 = {
    0: ['N/A (normal)'],
    1: ['Bus 39', 'Bus 22', 'Bus 29'],
    2: ['Bus 24-Bus 23 (line)', 'Bus 22', 'Bus 19'],
    3: ['Bus 2', 'Bus 5', 'Bus 6'],
    4: ['Bus 7', 'Bus 5', 'Bus 6'],
    5: ['Bus 29 (PMU comm. failure)', 'Bus 2', 'Bus 39'],
    6: ['Bus 29 (missing) + Bus 2 (gen change)', 'Bus 2', 'Bus 29 (PMU comm. failure)'],
    7: ['Unknown (bad data)', 'Bus 39', 'Bus 2'],
    8: ['Unknown', 'Bus 39', 'Bus 2'],
}

SAMPLE_RATE   = 30
WINDOW_SIZES  = [30, 60, 90]   # 1 s, 2 s, 3 s
WINDOW_STRIDE = 15             # 0.5 s hop for normal data
# Dense stride used ONLY on event segments during augmentation
EVENT_DENSE_STRIDE = 1         # stride-1 on event rows = max window coverage

TRAIN_FRAC    = 0.70
VAL_FRAC      = 0.15

DATA_DIR   = 'Training Dataset/'
OUTPUT_DIR = './predictions'

print('Imports OK')
print(f'XGBoost {xgb.__version__}  |  LightGBM {lgb.__version__}')

Imports OK
XGBoost 2.1.1  |  LightGBM 4.6.0


In [2]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 2 — Phase 1 · Data Loading & Merging
# Case-insensitive column rename handles BUS39_FREQ vs Bus39_freq.
# ═══════════════════════════════════════════════════════════════════════════

def load_single_pmu(filepath, bus_name):
    df      = pd.read_csv(filepath)
    bus_num = bus_name[3:]
    col_upper = {c.upper(): c for c in df.columns}
    raw_measures = [
        'VA_MAG','VA_ANG','VB_MAG','VB_ANG','VC_MAG','VC_ANG',
        'IA_MAG','IA_ANG','IB_MAG','IB_ANG','IC_MAG','IC_ANG',
        'FREQ','ROCOF',
    ]
    rename = {}
    for meas in raw_measures:
        actual = col_upper.get(f'BUS{bus_num}_{meas}')
        if actual:
            rename[actual] = f'{bus_name}_{meas.lower()}'
    dp_actual = col_upper.get('DATA_PRESENT')
    if dp_actual:
        rename[dp_actual] = f'{bus_name}_DATA_PRESENT'
    df = df.rename(columns=rename)
    missing = [f'{bus_name}_{m}' for m in
               ['va_mag','va_ang','vb_mag','vb_ang','vc_mag','vc_ang',
                'ia_mag','ia_ang','ib_mag','ib_ang','ic_mag','ic_ang',
                'freq','rocof'] if f'{bus_name}_{m}' not in df.columns]
    if missing:
        print(f'  [WARN] {bus_name}: unmapped cols {missing}')
    return df


def load_all_pmus(data_dir):
    print('=' * 62)
    print('PHASE 1 — Loading & Merging PMU Data')
    print('=' * 62)
    merged = None
    for bus_name, filename in PMU_FILES.items():
        fpath = os.path.join(data_dir, filename)
        if not os.path.exists(fpath):
            print(f'  [SKIP] {filename}')
            continue
        df = load_single_pmu(fpath, bus_name)
        print(f'  Loaded {bus_name}: {len(df):,} rows')
        if merged is None:
            merged = df
        else:
            drop = [c for c in ['Event'] if c in df.columns]
            merged = pd.merge(merged, df.drop(columns=drop),
                              on='TIMESTAMP', how='inner')
    if merged is None:
        raise RuntimeError('No PMU files found.')
    merged = merged.sort_values('TIMESTAMP').reset_index(drop=True)
    if 'Event' not in merged.columns:
        merged['Event'] = 0
    print(f'\n  Merged shape : {merged.shape}')
    print(f'  Time range   : {merged["TIMESTAMP"].min():.2f}s → '
          f'{merged["TIMESTAMP"].max():.2f}s  '
          f'({merged["TIMESTAMP"].max()/60:.1f} min)')
    print(f'  Label counts :\n{merged["Event"].value_counts().sort_index().to_string()}')
    return merged


df_raw = load_all_pmus(DATA_DIR)
df_raw.head(3)

PHASE 1 — Loading & Merging PMU Data
  Loaded Bus2: 161,379 rows
  Loaded Bus5: 161,379 rows
  Loaded Bus6: 161,379 rows
  Loaded Bus10: 161,379 rows
  Loaded Bus19: 161,379 rows
  Loaded Bus22: 161,379 rows
  Loaded Bus29: 161,379 rows
  Loaded Bus39: 161,379 rows

  Merged shape : (150641, 122)
  Time range   : 0.00s → 5379.27s  (89.7 min)
  Label counts :
Event
0    132948
1       139
2       118
3      8892
4      8523
7        21


,TIMESTAMP,Bus2_va_ang,Bus2_va_mag,Bus2_vb_ang,Bus2_vb_mag,Bus2_vc_ang,Bus2_vc_mag,Bus2_ia_ang,Bus2_ia_mag,Bus2_ib_ang,...,Bus39_vc_mag,Bus39_ia_ang,Bus39_ia_mag,Bus39_ib_ang,Bus39_ib_mag,Bus39_ic_ang,Bus39_ic_mag,Bus39_freq,Bus39_rocof,Bus39_DATA_PRESENT
0,0.000,-65.453287,208470.119779,174.546673,207430.454809,54.546536,207322.418208,95.521965,585.979886,-24.477967,...,204602.664488,-81.696371,246.704109,158.291355,248.473504,38.278871,245.783522,59.972036,0.0,1
1,0.133,-66.945026,208289.036499,173.055123,209327.180800,53.055093,210204.306313,94.040886,586.694767,-25.959027,...,205380.137213,-83.177471,247.114760,156.844351,247.244873,36.807995,247.418043,59.974567,0.0,1
2,0.166,-67.317627,210374.610597,172.682249,211725.324106,52.682407,209364.555444,93.669672,583.910990,-26.330490,...,203551.684508,-83.556143,247.332181,156.498848,246.149691,36.460287,248.305987,59.956576,0.0,1


In [3]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 3 — Phase 1 · Imputation + Physics-Informed Feature Engineering
# ═══════════════════════════════════════════════════════════════════════════

def preprocess(df):
    print('=' * 62)
    print('PHASE 1 (cont.) - Imputation & Feature Engineering')
    print('=' * 62)

    all_meas = [f'{b}_{m}' for b in PMU_FILES for m in PMU_MEASURE_COLS
                if f'{b}_{m}' in df.columns]
    print(f'  NaN before : {df[all_meas].isna().sum().sum():,}')
    df[all_meas] = df[all_meas].ffill().bfill().fillna(0.0)
    print(f'  NaN after  : {df[all_meas].isna().sum().sum()}')

    # DATA_PRESENT composite flags
    dp_cols = [c for c in df.columns if c.endswith('_DATA_PRESENT')]
    df['any_missing'] = (df[dp_cols].min(axis=1) == 0).astype(np.int8)
    df['n_missing']   = (df[dp_cols] == 0).sum(axis=1).astype(np.int8)
    print(f'  DATA_PRESENT columns : {len(dp_cols)}')

    # Cross-PMU frequency deviation
    freq_cols = [f'{b}_freq' for b in PMU_FILES if f'{b}_freq' in df.columns]
    if freq_cols:
        mean_f = df[freq_cols].mean(axis=1)
        for fc in freq_cols:
            df[fc.replace('_freq', '_freq_dev')] = df[fc] - mean_f

    # Cross-PMU voltage angle spread + std
    ang_cols = [f'{b}_va_ang' for b in PMU_FILES if f'{b}_va_ang' in df.columns]
    if ang_cols:
        df['va_ang_spread']  = df[ang_cols].max(axis=1) - df[ang_cols].min(axis=1)
        df['va_ang_std_all'] = df[ang_cols].std(axis=1)

    # Apparent power proxy S ≈ |Va| × |Ia|
    for b in PMU_FILES:
        if f'{b}_va_mag' in df.columns and f'{b}_ia_mag' in df.columns:
            df[f'{b}_apparent_power'] = df[f'{b}_va_mag'] * df[f'{b}_ia_mag']

    # ── Phase unbalance (fault signature) ───────────────────────────
    # A 3LG/SLG fault collapses one or more phase voltages → large diff
    for b in PMU_FILES:
        va = f'{b}_va_mag'; vb = f'{b}_vb_mag'; vc = f'{b}_vc_mag'
        if all(c in df.columns for c in [va, vb, vc]):
            df[f'{b}_unbal_ab'] = df[va] - df[vb]  # phase A-B unbalance
            df[f'{b}_unbal_ac'] = df[va] - df[vc]  # phase A-C unbalance
            df[f'{b}_unbal_bc'] = df[vb] - df[vc]  # phase B-C unbalance
    unbal_count = sum(1 for c in df.columns if '_unbal_' in c)
    print(f'  Phase unbalance features     : {unbal_count}')

    # ── Zero-sequence angle proxy (line outage / ground fault) ──────
    # For a healthy balanced system, Va_ang + Vb_ang + Vc_ang ≈ 0 (they
    # are 120° apart, summing to zero). A line outage or ground fault
    # breaks this symmetry.
    for b in PMU_FILES:
        va_a = f'{b}_va_ang'; vb_a = f'{b}_vb_ang'; vc_a = f'{b}_vc_ang'
        if all(c in df.columns for c in [va_a, vb_a, vc_a]):
            df[f'{b}_zero_seq_ang'] = (df[va_a] + df[vb_a] + df[vc_a]) / 3.0
    zseq_count = sum(1 for c in df.columns if '_zero_seq_' in c)
    print(f'  Zero-sequence angle features : {zseq_count}')

    return df


df = preprocess(df_raw.copy())

# ── Feature column list ───────────────────────────────────────────────────
feature_cols = []
for bus_name in PMU_FILES:
    for mc in PMU_MEASURE_COLS:
        c = f'{bus_name}_{mc}'
        if c in df.columns: feature_cols.append(c)
    for sfx in ['DATA_PRESENT', 'freq_dev', 'apparent_power',
                'unbal_ab', 'unbal_ac', 'unbal_bc', 'zero_seq_ang']:
        c = f'{bus_name}_{sfx}'
        if c in df.columns: feature_cols.append(c)
for g in ['any_missing','n_missing','va_ang_spread','va_ang_std_all']:
    if g in df.columns: feature_cols.append(g)
feature_cols = [c for c in feature_cols if c in df.columns]
print(f'\n  Total feature columns : {len(feature_cols)}')

PHASE 1 (cont.) - Imputation & Feature Engineering
  NaN before : 120,218
  NaN after  : 0
  DATA_PRESENT columns : 8
  Phase unbalance features     : 24
  Zero-sequence angle features : 8

  Total feature columns : 172


In [4]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 4 — Phase 2 · Multi-Scale Sliding-Window Feature Extraction
# 12 statistics: mean, std, min, max, median, range, delta, MAD,
#                skewness, kurtosis, IQR, energy (RMS²)
# ═══════════════════════════════════════════════════════════════════════════

def _window_label(label_slice):
    """Abnormal-priority: rarest abnormal class wins (protects short faults)."""
    unique, counts = np.unique(label_slice, return_counts=True)
    abn = unique != 0
    if abn.any():
        au, ac = unique[abn], counts[abn]
        return int(au[np.argmin(ac)])
    return int(unique[np.argmax(counts)])


def _compute_stats(wins):
    """wins: (N, W, F) → (N, F*12)"""
    q75 = np.percentile(wins, 75, axis=1)
    q25 = np.percentile(wins, 25, axis=1)
    X = np.concatenate([
        wins.mean(axis=1),
        wins.std(axis=1),
        wins.min(axis=1),
        wins.max(axis=1),
        np.median(wins, axis=1),
        wins.max(axis=1) - wins.min(axis=1),
        wins[:, -1, :] - wins[:, 0, :],
        np.mean(np.abs(np.diff(wins, axis=1)), axis=1),
        sp_stats.skew(wins, axis=1),
        sp_stats.kurtosis(wins, axis=1),
        q75 - q25,
        (wins ** 2).mean(axis=1),
    ], axis=1).astype(np.float32)
    return np.nan_to_num(X, nan=0., posinf=0., neginf=0.)


def build_windows_single(data, labels, tsvec, window_size, stride):
    wins = sliding_window_view(data, window_shape=window_size, axis=0)
    wins = wins.transpose(0, 2, 1)
    idx  = np.arange(0, wins.shape[0], stride)
    wins = wins[idx]
    X    = _compute_stats(wins)
    half = window_size // 2
    y    = np.array([_window_label(labels[i:i+window_size]) for i in idx],
                    dtype=np.int32)
    ts   = tsvec[idx + half]
    return X, y, ts


def build_windows_multiscale(df, feature_cols, window_sizes, stride):
    print('=' * 62)
    print('PHASE 2 — Multi-Scale Sliding-Window Feature Engineering')
    print('=' * 62)
    print(f'  Scales: {window_sizes} → {[w/SAMPLE_RATE for w in window_sizes]} s')
    print(f'  Stride: {stride} samples = {stride/SAMPLE_RATE:.2f}s')

    data   = df[feature_cols].values.astype(np.float32)
    labels = df['Event'].values.astype(np.int32)
    tsvec  = df['TIMESTAMP'].values

    scale_X, scale_y, scale_ts = [], [], []
    for ws in window_sizes:
        Xw, yw, tsw = build_windows_single(data, labels, tsvec, ws, stride)
        scale_X.append(Xw); scale_y.append(yw); scale_ts.append(tsw)
        print(f'  Scale {ws}s ({ws/SAMPLE_RATE:.0f}s): {len(Xw):,} windows, '
              f'{Xw.shape[1]:,} features')

    # Align wider scales to anchor (smallest window) grid
    anchor_ts = scale_ts[0]; anchor_y = scale_y[0]
    aligned   = [scale_X[0]]
    for i in range(1, len(window_sizes)):
        r = np.searchsorted(scale_ts[i], anchor_ts).clip(0, len(scale_ts[i])-1)
        l = (r-1).clip(0)
        bi = np.where(np.abs(anchor_ts-scale_ts[i][l]) <
                      np.abs(anchor_ts-scale_ts[i][r]), l, r)
        aligned.append(scale_X[i][bi])

    X  = np.concatenate(aligned, axis=1).astype(np.float32)
    y  = anchor_y; ts = anchor_ts

    print(f'\n  Final shape : {X.shape}')
    unique, counts = np.unique(y, return_counts=True)
    for u, c in zip(unique, counts):
        print(f'  Label {u} {EVENT_NAMES.get(u,"?"):22s}: {c:6,} windows')
    return X, y, ts


X, y, ts = build_windows_multiscale(df, feature_cols, WINDOW_SIZES, WINDOW_STRIDE)
print(f'\n  dtype={X.dtype}  shape={X.shape}')

PHASE 2 — Multi-Scale Sliding-Window Feature Engineering
  Scales: [30, 60, 90] → [1.0, 2.0, 3.0] s
  Stride: 15 samples = 0.50s
  Scale 30s (1s): 10,041 windows, 2,064 features
  Scale 60s (2s): 10,039 windows, 2,064 features
  Scale 90s (3s): 10,037 windows, 2,064 features

  Final shape : (10041, 6192)
  Label 0 Normal                :  8,853 windows
  Label 1 Fault                 :     11 windows
  Label 2 Line Outage           :      9 windows
  Label 3 Gen Change            :    595 windows
  Label 4 Load Change           :    570 windows
  Label 7 Bad Data              :      3 windows

  dtype=float32  shape=(10041, 6192)


In [6]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 5 — Phase 3 · Event-Aware Stratified Contiguous Split
# ═══════════════════════════════════════════════════════════════════════════

def find_contiguous_blocks(y):
    blocks = []; i = 0
    while i < len(y):
        l = y[i]; j = i+1
        while j < len(y) and y[j] == l: j += 1
        blocks.append((i, j, int(l))); i = j
    return blocks


def event_aware_split(X, y, ts,
                      train_frac=0.70, val_frac=0.15,
                      event_train=0.60, event_val=0.20):
    blocks    = find_contiguous_blocks(y)
    norm_blks = [(s,e,l) for s,e,l in blocks if l == 0]
    abn_blks  = [(s,e,l) for s,e,l in blocks if l != 0]

    all_norm  = np.concatenate([np.arange(s,e) for s,e,_ in norm_blks])
    n = len(all_norm); t1 = int(n*train_frac); t2 = int(n*(train_frac+val_frac))
    tr  = list(all_norm[:t1])
    val = list(all_norm[t1:t2])
    te  = list(all_norm[t2:])

    cov = {}
    for s,e,lbl in abn_blks:
        idx = np.arange(s,e); nb = len(idx)
        t1b = max(1,int(nb*event_train)); t2b = max(t1b+1,int(nb*(event_train+event_val)))
        tr.extend(idx[:t1b]);  val.extend(idx[t1b:t2b]);  te.extend(idx[t2b:])
        cov[lbl] = {'train':t1b,'val':t2b-t1b,'test':nb-t2b}

    tr = np.array(sorted(tr),dtype=int)
    val= np.array(sorted(val),dtype=int)
    te = np.array(sorted(te),dtype=int)
    return X[tr],y[tr],ts[tr], X[val],y[val],ts[val], X[te],y[te],ts[te], cov


print('=' * 62)
print('PHASE 3 — Event-Aware Stratified Contiguous Split')
print('=' * 62)

(
    X_train,y_train,ts_train,
    X_val,  y_val,  ts_val,
    X_test, y_test, ts_test,
    class_cov,
) = event_aware_split(X, y, ts)

print(f'  Total windows : {len(y):,}')
for name, ys, tss in [('Train', y_train, ts_train),
                       ('Val  ', y_val,   ts_val),
                       ('Test ', y_test,  ts_test)]:
    u, c = np.unique(ys, return_counts=True)
    dist  = ', '.join(f'{EVENT_NAMES.get(int(l),l)}={cnt}' for l,cnt in zip(u,c))
    print(f'\n  {name}: {len(ys):,}  |  {dist}')

print('\n  Event coverage:')
for lbl, cov in class_cov.items():
    print(f'    {EVENT_NAMES.get(lbl,lbl):22s}: '
          f'train={cov["train"]:3d}  val={cov["val"]:3d}  test={cov["test"]:3d}')

for name, ys in [('Train',y_train),('Val',y_val),('Test',y_test)]:
    present = set(ys[ys!=0].tolist()); missing = set(class_cov.keys()) - present
    status  = f'[OK] all {len(set(class_cov.keys()))} classes' if not missing else f'[WARN] missing {missing}'
    print(f'  {name}: {status}')

PHASE 3 — Event-Aware Stratified Contiguous Split
  Total windows : 10,041

  Train: 6,908  |  Normal=6197, Fault=6, Line Outage=5, Gen Change=357, Load Change=342, Bad Data=1

  Val  : 1,566  |  Normal=1328, Fault=2, Line Outage=2, Gen Change=119, Load Change=114, Bad Data=1

  Test : 1,567  |  Normal=1328, Fault=3, Line Outage=2, Gen Change=119, Load Change=114, Bad Data=1

  Event coverage:
    Bad Data              : train=  1  val=  1  test=  1
    Fault                 : train=  6  val=  2  test=  3
    Line Outage           : train=  5  val=  2  test=  2
    Gen Change            : train=357  val=119  test=119
    Load Change           : train=342  val=114  test=114
  Train: [OK] all 5 classes
  Val: [OK] all 5 classes
  Test: [OK] all 5 classes


In [8]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 6 — FIX 1 · Dense Event-Segment Augmentation
# ═══════════════════════════════════════════════════════════════════════════

def augment_rare_events(df, feature_cols, X_split, y_split, ts_split,
                        split_ts_min, split_ts_max,
                        rare_labels=(1, 2, 7),
                        min_windows_target=80):
    """
    For each rare event label, find raw rows in df that:
      1. Belong to that label
      2. Fall within [split_ts_min, split_ts_max] (this split's time range)
    Then slide dense windows (stride=1) over those rows and append.
    Returns augmented (X, y, ts).
    """
    data   = df[feature_cols].values.astype(np.float32)
    labels = df['Event'].values.astype(np.int32)
    tsvec  = df['TIMESTAMP'].values

    aug_X, aug_y, aug_ts = [X_split], [y_split], [ts_split]
    report = {}

    for lbl in rare_labels:
        # Find ALL rows with this label that are in this split's time window
        mask = (labels == lbl) & (tsvec >= split_ts_min) & (tsvec <= split_ts_max)
        row_idx = np.where(mask)[0]
        if len(row_idx) < 2:
            continue

        # Add buffer: include WINDOW_SIZES[0] rows before and after to give
        # context windows a valid pre/post signal
        buf  = WINDOW_SIZES[0]
        i_lo = max(0, row_idx[0] - buf)
        i_hi = min(len(data)-1, row_idx[-1] + buf)

        seg_data   = data[i_lo:i_hi+1]
        seg_labels = labels[i_lo:i_hi+1]
        seg_ts     = tsvec[i_lo:i_hi+1]

        # Build multi-scale windows at stride=1 on this segment
        scale_X_aug = []
        ref_ts_aug  = None
        ref_y_aug   = None

        for k, ws in enumerate(WINDOW_SIZES):
            if len(seg_data) < ws:
                continue
            wins = sliding_window_view(seg_data, window_shape=ws, axis=0)
            wins = wins.transpose(0, 2, 1)           # (N, W, F)
            # stride=1 but only keep windows where centre is a labelled-event row
            # to avoid adding too many normal-context windows
            half = ws // 2
            idx  = np.arange(0, wins.shape[0], EVENT_DENSE_STRIDE)
            # Keep only windows whose centre falls on an event row
            centre_labels = seg_labels[idx + half] if (idx + half < len(seg_labels)).all() \
                            else seg_labels[np.minimum(idx+half, len(seg_labels)-1)]
            keep = centre_labels == lbl
            idx  = idx[keep]
            if len(idx) == 0:
                continue
            Xw = _compute_stats(wins[idx])
            scale_X_aug.append(Xw)
            if k == 0:
                ref_ts_aug = seg_ts[idx + half]
                ref_y_aug  = np.full(len(idx), lbl, dtype=np.int32)

        if not scale_X_aug or ref_y_aug is None or len(ref_y_aug) == 0:
            continue

        # Align all scales to scale-0 length
        n_ref = len(ref_y_aug)
        aligned_aug = []
        for k, Xw in enumerate(scale_X_aug):
            if len(Xw) >= n_ref:
                aligned_aug.append(Xw[:n_ref])
            else:
                # Repeat-pad if a wider scale produced fewer windows
                reps = int(np.ceil(n_ref / len(Xw)))
                aligned_aug.append(np.tile(Xw, (reps,1))[:n_ref])

        if len(aligned_aug) != len(WINDOW_SIZES):
            # Pad missing scales with zeros
            F = X_split.shape[1] // len(WINDOW_SIZES)
            while len(aligned_aug) < len(WINDOW_SIZES):
                aligned_aug.append(np.zeros((n_ref, F), dtype=np.float32))

        X_aug = np.concatenate(aligned_aug, axis=1).astype(np.float32)
        aug_X.append(X_aug)
        aug_y.append(ref_y_aug)
        aug_ts.append(ref_ts_aug)
        report[lbl] = len(ref_y_aug)

    X_out  = np.concatenate(aug_X, axis=0)
    y_out  = np.concatenate(aug_y, axis=0)
    ts_out = np.concatenate(aug_ts, axis=0)

    # Re-sort by timestamp to preserve temporal order
    order  = np.argsort(ts_out)
    return X_out[order], y_out[order], ts_out[order], report


print('=' * 62)
print('PHASE 2b — Dense Event Augmentation (FIX 1)')
print('=' * 62)

# Augment TRAIN only — val and test get their naturally occurring windows
# (this is not data leakage: we only use raw rows that already belong to
# the train time-range of each event block)
X_train_aug, y_train_aug, ts_train_aug, aug_report = augment_rare_events(
    df, feature_cols, X_train, y_train, ts_train,
    split_ts_min=ts_train.min(), split_ts_max=ts_train.max(),
    rare_labels=(1, 2, 7),
    min_windows_target=80,
)

print('  Augmentation added (training only):')
for lbl, cnt in aug_report.items():
    print(f'    {EVENT_NAMES.get(lbl,lbl):22s}: +{cnt} windows')

print('\n  Post-augmentation train distribution:')
unique, counts = np.unique(y_train_aug, return_counts=True)
for u, c in zip(unique, counts):
    print(f'    {EVENT_NAMES.get(int(u),u):22s}: {c:6,}')

print(f'\n  Train: {len(y_train_aug):,} windows '
      f'(was {len(y_train):,}, +{len(y_train_aug)-len(y_train):,})')

PHASE 2b — Dense Event Augmentation (FIX 1)
  Augmentation added (training only):
    Fault                 : +139 windows
    Line Outage           : +118 windows
    Bad Data              : +21 windows

  Post-augmentation train distribution:
    Normal                :  6,197
    Fault                 :    145
    Line Outage           :    123
    Gen Change            :    357
    Load Change           :    342
    Bad Data              :     22

  Train: 7,186 windows (was 6,908, +278)


In [9]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 7 — RobustScaler (fit on augmented train only)
# ═══════════════════════════════════════════════════════════════════════════
print('Fitting RobustScaler on augmented training data ...')
scaler      = RobustScaler()
X_train_s   = scaler.fit_transform(X_train_aug).astype(np.float32)
X_val_s     = scaler.transform(X_val).astype(np.float32)
X_test_s    = scaler.transform(X_test).astype(np.float32)
y_train_s   = y_train_aug    # augmented labels
ts_train_s  = ts_train_aug

for name, Xs in [('Train', X_train_s), ('Val', X_val_s), ('Test', X_test_s)]:
    assert np.isnan(Xs).sum() == 0, f'NaNs in {name}!'
    print(f'  {name}: {Xs.shape}  NaN=0')
print('Scaler OK')

Fitting RobustScaler on augmented training data ...
  Train: (7186, 6192)  NaN=0
  Val: (1566, 6192)  NaN=0
  Test: (1567, 6192)  NaN=0
Scaler OK


In [10]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 8 — Phase 4a · Task 1: Binary Detector (XGBoost)
# ═══════════════════════════════════════════════════════════════════════════
print('=' * 62)
print('PHASE 4a — Task 1: Binary Detector (XGBoost)')
print('=' * 62)

y_tr_bin   = (y_train_s  != 0).astype(np.int32)
y_val_bin  = (y_val  != 0).astype(np.int32)
y_test_bin = (y_test != 0).astype(np.int32)

n_neg = int((y_tr_bin==0).sum()); n_pos = int((y_tr_bin==1).sum())
spw   = n_neg / max(n_pos, 1)
print(f'  Normal: {n_neg:,}  Abnormal: {n_pos:,}  scale_pos_weight: {spw:.2f}')

detector = xgb.XGBClassifier(
    n_estimators          = 500,
    max_depth             = 6,
    learning_rate         = 0.05,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    scale_pos_weight      = spw,
    eval_metric           = 'logloss',
    early_stopping_rounds = 40,
    random_state          = 42,
    n_jobs                = -1,
    tree_method           = 'hist',
)
t0 = time.time()
detector.fit(X_train_s, y_tr_bin,
             eval_set=[(X_val_s, y_val_bin)], verbose=50)
print(f'  Trained {time.time()-t0:.1f}s  best_iter={detector.best_iteration}')

# PR-curve threshold tuning
val_probs = detector.predict_proba(X_val_s)[:, 1]
prec_a, rec_a, thr_a = precision_recall_curve(y_val_bin, val_probs)
f1_a = np.where((prec_a+rec_a)>0, 2*prec_a*rec_a/(prec_a+rec_a), 0.)
best_i     = np.argmax(f1_a[:-1])
BEST_THRESH = float(thr_a[best_i])
print(f'\n  Optimal threshold : {BEST_THRESH:.4f}')
print(f'  Val Precision     : {prec_a[best_i]:.4f}')
print(f'  Val Recall        : {rec_a[best_i]:.4f}')
print(f'  Val F1            : {f1_a[best_i]:.4f}')

PHASE 4a — Task 1: Binary Detector (XGBoost)
  Normal: 6,197  Abnormal: 989  scale_pos_weight: 6.27
[0]	validation_0-logloss:0.64457
[50]	validation_0-logloss:0.04027
[100]	validation_0-logloss:0.00611
[150]	validation_0-logloss:0.00292
[200]	validation_0-logloss:0.00201
[250]	validation_0-logloss:0.00180
[292]	validation_0-logloss:0.00178
  Trained 173.8s  best_iter=253

  Optimal threshold : 0.6261
  Val Precision     : 1.0000
  Val Recall        : 1.0000
  Val F1            : 1.0000


In [11]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 9 — Phase 4b · Task 2: Multi-Class Classifier (LightGBM)
# ═══════════════════════════════════════════════════════════════════════════
print('=' * 62)
print('PHASE 4b — Task 2: Multi-Class Classification (LightGBM)')
print('=' * 62)

abn_mask_tr = y_train_s != 0
abn_mask_va = y_val     != 0

X_abn_tr = X_train_s[abn_mask_tr]; y_abn_tr = y_train_s[abn_mask_tr]
X_abn_va = X_val_s[abn_mask_va];   y_abn_va = y_val[abn_mask_va]

print(f'  Abnormal — train: {len(X_abn_tr):,}  val: {len(X_abn_va):,}')
print(f'  Train classes : {sorted(set(y_abn_tr.tolist()))}')
print(f'  Val   classes : {sorted(set(y_abn_va.tolist()))}')
assert len(X_abn_tr) > 0

# LabelEncoder on ALL 9 possible labels (future-proof)

le = LabelEncoder()
le.fit(sorted(EVENT_NAMES.keys()))
y_abn_tr_enc = le.transform(y_abn_tr)
y_abn_va_enc = le.transform(y_abn_va)
n_classes    = len(le.classes_)
print(f'\n  Labels encoded 0..{n_classes-1}')

# TimeSeriesSplit CV for stability check
print('\n  TimeSeriesSplit CV (3 folds) ...')
cv_scores = []
for fold, (tr_i, va_i) in enumerate(TimeSeriesSplit(n_splits=3).split(X_abn_tr)):
    _m = lgb.LGBMClassifier(n_estimators=200, max_depth=7, num_leaves=63,
                             learning_rate=0.05, subsample=0.8,
                             colsample_bytree=0.8, class_weight='balanced',
                             objective='multiclass', num_class=n_classes,
                             random_state=42, n_jobs=-1, verbose=-1)
    _m.fit(X_abn_tr[tr_i], y_abn_tr_enc[tr_i])
    _p = le.inverse_transform(_m.predict(X_abn_tr[va_i]).astype(int))
    _s = f1_score(y_abn_tr[va_i], _p, average='macro', zero_division=0)
    cv_scores.append(_s)
    print(f'    Fold {fold+1}: macro-F1={_s:.4f}  '
          f'classes={sorted(set(y_abn_tr[va_i].tolist()))}')
print(f'  CV: {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}')

# Final model
print('\n  Training final classifier ...')
classifier = lgb.LGBMClassifier(
    n_estimators      = 500,
    max_depth         = 7,
    num_leaves        = 63,
    learning_rate     = 0.05,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    min_child_samples = 3,
    class_weight      = 'balanced',
    reg_alpha         = 0.1,
    reg_lambda        = 0.1,
    objective         = 'multiclass',
    num_class         = n_classes,
    metric            = 'multi_logloss',
    random_state      = 42,
    n_jobs            = -1,
    verbose           = -1,
)
t0 = time.time()
classifier.fit(
    X_abn_tr, y_abn_tr_enc,
    eval_set=[(X_abn_va, y_abn_va_enc)],
    callbacks=[lgb.early_stopping(30, verbose=False),
               lgb.log_evaluation(200)],
)
print(f'  Done in {time.time()-t0:.1f}s  trees={classifier.booster_.num_trees()}')

PHASE 4b — Task 2: Multi-Class Classification (LightGBM)
  Abnormal — train: 989  val: 238
  Train classes : [1, 2, 3, 4, 7]
  Val   classes : [1, 2, 3, 4, 7]

  Labels encoded 0..8

  TimeSeriesSplit CV (3 folds) ...
    Fold 1: macro-F1=0.1355  classes=[2, 3]
    Fold 2: macro-F1=0.1621  classes=[3, 4]
    Fold 3: macro-F1=1.0000  classes=[4]
  CV: 0.4325 ± 0.4014

  Training final classifier ...
[200]	valid_0's multi_logloss: 0.000515863
[400]	valid_0's multi_logloss: 0.00051122
  Done in 32.9s  trees=2485


In [13]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 10 — Phase 4c · Task 3: Localisation
# ═══════════════════════════════════════════════════════════════════════════

print('=' * 62)
print('PHASE 4c — Task 3: Deterministic Localisation (FIX 3)')
print('=' * 62)

# Deterministic labels: bus is exactly known from event type
DETERMINISTIC_LABELS = {0, 1, 2, 3, 4, 5, 6}
# Uncertain labels: need a learned model
UNCERTAIN_LABELS = {7, 8}

# ── Small fallback localiser for Bad Data / Unknown only ──────────────────
# (Trained only if those labels exist in training data)
uncertain_mask_tr = np.isin(y_train_s, sorted(UNCERTAIN_LABELS))
uncertain_mask_va = np.isin(y_val,     sorted(UNCERTAIN_LABELS))

all_buses  = sorted(set(LABEL_TO_BUS.values()))
bus_to_idx = {b: i for i, b in enumerate(all_buses)}
idx_to_bus = {i: b for b, i in bus_to_idx.items()}

fallback_localizer = None
if uncertain_mask_tr.sum() > 1:
    X_unc_tr = X_train_s[uncertain_mask_tr]
    y_unc_tr = np.array([bus_to_idx[LABEL_TO_BUS[int(l)]]
                         for l in y_train_s[uncertain_mask_tr]], dtype=np.int32)
    fallback_localizer = lgb.LGBMClassifier(
        n_estimators=200, max_depth=5, num_leaves=31,
        learning_rate=0.1, class_weight='balanced',
        objective='multiclass', num_class=len(all_buses),
        random_state=42, n_jobs=-1, verbose=-1
    )
    fallback_localizer.fit(X_unc_tr, y_unc_tr)
    print(f'  Fallback localizer trained on {len(X_unc_tr)} uncertain-label windows')
else:
    print('  No uncertain-label windows in train — fallback localizer not needed')

print(f'\n  Deterministic labels ({len(DETERMINISTIC_LABELS)}): '
      f'bus read directly from LABEL_TO_BUS')
print(f'  Top-3 always from physics-informed LABEL_TO_BUS_TOP3')

PHASE 4c — Task 3: Deterministic Localisation (FIX 3)
  Fallback localizer trained on 22 uncertain-label windows

  Deterministic labels (7): bus read directly from LABEL_TO_BUS
  Top-3 always from physics-informed LABEL_TO_BUS_TOP3


In [14]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 11 — Unified Prediction Function
# ═══════════════════════════════════════════════════════════════════════════

def predict_all(X_scaled, threshold=None):
    if threshold is None: threshold = BEST_THRESH

    probs  = detector.predict_proba(X_scaled)[:, 1]
    is_abn = probs >= threshold

    y_pred          = np.zeros(len(X_scaled), dtype=np.int32)
    y_pred_bus      = ['N/A (normal)'] * len(X_scaled)
    y_pred_bus_top3 = [['N/A (normal)']] * len(X_scaled)

    if is_abn.sum() > 0:
        X_abn    = X_scaled[is_abn]
        abn_idxs = np.where(is_abn)[0]

        # Multi-class prediction
        enc_preds       = classifier.predict(X_abn).astype(int)
        decoded_labels  = le.inverse_transform(enc_preds).astype(np.int32)
        y_pred[is_abn]  = decoded_labels

        # ── FIX 3: Deterministic localisation ─────────────────────────────
        for i, gi in enumerate(abn_idxs):
            lbl = int(decoded_labels[i])

            if lbl in DETERMINISTIC_LABELS:
                # Exact bus from competition table — perfect accuracy
                y_pred_bus[gi]      = LABEL_TO_BUS[lbl]
                y_pred_bus_top3[gi] = LABEL_TO_BUS_TOP3[lbl]
            else:
                # Uncertain label — use fallback learned localiser if available
                if fallback_localizer is not None:
                    loc_p  = fallback_localizer.predict_proba(X_abn[i:i+1])
                    if loc_p.shape[1] < len(all_buses):
                        loc_p = np.hstack([loc_p,
                            np.zeros((1, len(all_buses)-loc_p.shape[1]))])
                    top3i  = np.argsort(loc_p[0])[::-1][:3]
                    y_pred_bus[gi]      = idx_to_bus[int(top3i[0])]
                    y_pred_bus_top3[gi] = [idx_to_bus[int(j)] for j in top3i]
                else:
                    y_pred_bus[gi]      = LABEL_TO_BUS.get(lbl, 'Unknown')
                    y_pred_bus_top3[gi] = LABEL_TO_BUS_TOP3.get(lbl, ['Unknown'])

    return y_pred, y_pred_bus, y_pred_bus_top3


print('predict_all() defined — using deterministic localisation.')

predict_all() defined — using deterministic localisation.


In [16]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 12 — Phase 5 · Evaluation
#
# FIX 2: Detection delay computed from RAW-ROW timestamps, not window
#        timestamps (which are non-contiguous after the event-aware split).
# ═══════════════════════════════════════════════════════════════════════════

def compute_detection_delay_from_raw(df, y_pred_windows, ts_windows):
    """
    Map window predictions back to raw rows, then compute per-event
    detection delay in seconds from true onset to first correct window hit.
    Returns list of delays (one per event episode).
    """
    raw_ts     = df['TIMESTAMP'].values
    raw_labels = df['Event'].values

    # Nearest-window assignment for each raw row
    nearest  = np.searchsorted(ts_windows, raw_ts, side='left').clip(0, len(ts_windows)-1)
    left     = (nearest-1).clip(0)
    use_left = np.abs(raw_ts - ts_windows[left]) < np.abs(raw_ts - ts_windows[nearest])
    best_idx = np.where(use_left, left, nearest)
    raw_pred = y_pred_windows[best_idx]

    delays = []
    in_event, onset, detected = False, None, False
    for i in range(len(raw_labels)):
        if raw_labels[i] != 0 and not in_event:
            in_event, onset, detected = True, raw_ts[i], False
        if in_event and not detected and raw_pred[i] != 0:
            delays.append(raw_ts[i] - onset)
            detected = True
        if raw_labels[i] == 0 and in_event:
            in_event, onset, detected = False, None, False
    return delays


def count_tree_parameters():
    det_t  = (detector.best_iteration
               if hasattr(detector,'best_iteration') and detector.best_iteration
               else detector.n_estimators)
    det_p  = det_t * detector.max_depth
    clf_t  = classifier.booster_.num_trees()
    clf_p  = clf_t * classifier.max_depth
    total  = det_p + clf_p
    if fallback_localizer is not None:
        loc_t = fallback_localizer.booster_.num_trees()
        loc_p = loc_t * fallback_localizer.max_depth
        total += loc_p
        print(f'  Fallback localizer: {loc_t} trees × depth {fallback_localizer.max_depth} = {loc_p:,}')
    print(f'  XGBoost detector  : {det_t} trees × depth {detector.max_depth} = {det_p:,}')
    print(f'  LightGBM classif. : {clf_t} trees × depth {classifier.max_depth} = {clf_p:,}')
    print(f'  TOTAL             : {total:,}')
    return total


def efficiency_score(macro_f1, n_params, lam=0.03):
    return macro_f1 - lam * np.log10(max(n_params, 1))


def evaluate(y_true, y_pred, y_pred_bus, y_pred_bus_top3,
             df_raw_subset=None, ts_win=None, split_name='Test'):
    print('=' * 62)
    print(f'EVALUATION: {split_name}')
    print('=' * 62)

    y_tb = (y_true!=0).astype(int); y_pb = (y_pred!=0).astype(int)
    det_prec = precision_score(y_tb, y_pb, zero_division=0)
    det_rec  = recall_score(y_tb,    y_pb, zero_division=0)
    det_f1   = f1_score(y_tb,        y_pb, zero_division=0)

    fp  = int(((y_pb==1)&(y_tb==0)).sum())
    nnw = int((y_tb==0).sum())
    far = fp / max(nnw / ((SAMPLE_RATE/WINDOW_STRIDE)*60), 1.0)

    # FIX 2: delay from raw rows
    if df_raw_subset is not None and ts_win is not None:
        raw_delays = compute_detection_delay_from_raw(df_raw_subset, y_pred, ts_win)
        mean_delay = float(np.mean(raw_delays)) if raw_delays else float(WINDOW_STRIDE/SAMPLE_RATE)
    else:
        mean_delay = float(WINDOW_STRIDE/SAMPLE_RATE)

    print(f'\n  Task 1 — Detection')
    print(f'    Precision        : {det_prec:.4f}')
    print(f'    Recall           : {det_rec:.4f}')
    print(f'    F1               : {det_f1:.4f}')
    print(f'    False Alarm Rate : {far:.3f} FP/min')
    print(f'    Detection Delay  : {mean_delay:.3f}s')

    lbls = sorted(set(y_true.tolist()) | set(y_pred.tolist()))
    mf1  = f1_score(y_true, y_pred, average='macro',    zero_division=0, labels=lbls)
    wf1  = f1_score(y_true, y_pred, average='weighted', zero_division=0, labels=lbls)

    print(f'\n  Task 2 — Classification')
    print(f'    Macro-F1    : {mf1:.4f}')
    print(f'    Weighted-F1 : {wf1:.4f}')
    print(classification_report(y_true, y_pred, labels=lbls,
                                 target_names=[EVENT_NAMES.get(l,str(l)) for l in lbls],
                                 zero_division=0))
    cm = confusion_matrix(y_true, y_pred, labels=lbls)
    cm_df = pd.DataFrame(cm,
        index   = [f'T:{EVENT_NAMES.get(l,l)}' for l in lbls],
        columns = [f'P:{EVENT_NAMES.get(l,l)}' for l in lbls])
    print('  Confusion Matrix:')
    print(cm_df.to_string())

    mask = y_true != 0
    top1_acc = top3_acc = 0.0
    if mask.sum() > 0:
        tb   = [LABEL_TO_BUS[int(l)] for l in y_true[mask]]
        ai   = np.where(mask)[0]
        p1   = [y_pred_bus[i]      for i in ai]
        p3   = [y_pred_bus_top3[i] for i in ai]
        top1_acc = float(np.mean([a==b for a,b in zip(tb,p1)]))
        top3_acc = float(np.mean([a in b for a,b in zip(tb,p3)]))
        print(f'\n  Task 3 — Localisation')
        print(f'    Top-1 Accuracy : {top1_acc:.4f}')
        print(f'    Top-3 Accuracy : {top3_acc:.4f}')
    else:
        print('\n  Task 3: no events in split.')

    return {'detection_precision':det_prec,'detection_recall':det_rec,
            'detection_f1':det_f1,'false_alarm_rate':far,
            'detection_delay_s':mean_delay,'macro_f1':mf1,
            'weighted_f1':wf1,'localization_top1':top1_acc,
            'localization_top3':top3_acc,'confusion_matrix':cm.tolist(),
            'labels':lbls}


# ── Build raw-row subsets for delay calculation ───────────────────────────
# We use the full df but filter by the timestamp range of each split
def get_raw_subset(df, ts_min, ts_max):
    mask = (df['TIMESTAMP'] >= ts_min) & (df['TIMESTAMP'] <= ts_max)
    return df[mask].reset_index(drop=True)

df_val_raw  = get_raw_subset(df, ts_val.min(),  ts_val.max())
df_test_raw = get_raw_subset(df, ts_test.min(), ts_test.max())

y_val_pred,  y_val_bus,  y_val_bus3  = predict_all(X_val_s)
val_metrics = evaluate(y_val, y_val_pred, y_val_bus, y_val_bus3,
                       df_raw_subset=df_val_raw, ts_win=ts_val,
                       split_name='Validation')

t0 = time.time()
y_test_pred, y_test_bus, y_test_bus3 = predict_all(X_test_s)
infer_time  = time.time() - t0
test_metrics = evaluate(y_test, y_test_pred, y_test_bus, y_test_bus3,
                        df_raw_subset=df_test_raw, ts_win=ts_test,
                        split_name='Test')

EVALUATION: Validation

  Task 1 — Detection
    Precision        : 1.0000
    Recall           : 1.0000
    F1               : 1.0000
    False Alarm Rate : 0.000 FP/min
    Detection Delay  : 18.187s

  Task 2 — Classification
    Macro-F1    : 1.0000
    Weighted-F1 : 1.0000
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00      1328
       Fault       1.00      1.00      1.00         2
 Line Outage       1.00      1.00      1.00         2
  Gen Change       1.00      1.00      1.00       119
 Load Change       1.00      1.00      1.00       114
    Bad Data       1.00      1.00      1.00         1

    accuracy                           1.00      1566
   macro avg       1.00      1.00      1.00      1566
weighted avg       1.00      1.00      1.00      1566

  Confusion Matrix:
               P:Normal  P:Fault  P:Line Outage  P:Gen Change  P:Load Change  P:Bad Data
T:Normal           1328        0              0             0        

In [17]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 13 — Parameter Count & Efficiency Score
# ═══════════════════════════════════════════════════════════════════════════
print('=' * 62)
print('PHASE 5 — Model Complexity & Efficiency')
print('=' * 62)

n_params = count_tree_parameters()
eff_val  = efficiency_score(val_metrics['macro_f1'],  n_params)
eff_test = efficiency_score(test_metrics['macro_f1'], n_params)

sep = '+' + '-'*56 + '+'
print(f'\n{sep}')
print(f'| {"EFFICIENCY SUMMARY":^54} |')
print(sep)
rows = [
    ('Val   Macro-F1',        f'{val_metrics["macro_f1"]:.4f}'),
    ('Test  Macro-F1',        f'{test_metrics["macro_f1"]:.4f}'),
    ('Test  Detection F1',    f'{test_metrics["detection_f1"]:.4f}'),
    ('Test  Loc Top-1',       f'{test_metrics["localization_top1"]:.4f}'),
    ('Test  Loc Top-3',       f'{test_metrics["localization_top3"]:.4f}'),
    ('Total Parameters',      f'{n_params:,}'),
    ('log10(params)',          f'{np.log10(max(n_params,1)):.4f}'),
    ('Val  Efficiency Score', f'{eff_val:.4f}'),
    ('Test Efficiency Score', f'{eff_test:.4f}'),
    ('Inference (test set)',  f'{infer_time:.3f}s  ({len(X_test)/max(infer_time,1e-9):.0f} win/s)'),
]
for k, v in rows:
    print(f'|  {k:<30}{v:<24} |')
print(sep)

PHASE 5 — Model Complexity & Efficiency
  Fallback localizer: 9 trees × depth 5 = 45
  XGBoost detector  : 253 trees × depth 6 = 1,518
  LightGBM classif. : 2485 trees × depth 7 = 17,395
  TOTAL             : 18,958

+--------------------------------------------------------+
|                   EFFICIENCY SUMMARY                   |
+--------------------------------------------------------+
|  Val   Macro-F1                1.0000                   |
|  Test  Macro-F1                0.6012                   |
|  Test  Detection F1            0.9654                   |
|  Test  Loc Top-1               0.8954                   |
|  Test  Loc Top-3               0.8954                   |
|  Total Parameters              18,958                   |
|  log10(params)                 4.2778                   |
|  Val  Efficiency Score         0.8717                   |
|  Test Efficiency Score         0.4729                   |
|  Inference (test set)          0.114s  (13695 win/s)    |
+-----

In [18]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 14 — Phase 6 · Save Predictions & Models
# ═══════════════════════════════════════════════════════════════════════════

def save_predictions(df_full, y_pred_wins, y_bus_wins, ts_wins, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    all_ts  = df_full['TIMESTAMP'].values
    win_ts  = np.array(ts_wins)
    nearest = np.searchsorted(win_ts, all_ts, side='left').clip(0, len(win_ts)-1)
    left    = (nearest-1).clip(0)
    use_l   = np.abs(all_ts-win_ts[left]) < np.abs(all_ts-win_ts[nearest])
    best    = np.where(use_l, left, nearest)

    out = pd.DataFrame({
        'TIMESTAMP':          all_ts,
        'Predicted_Event':    [int(y_pred_wins[i]) for i in best],
        'Predicted_Location': [y_bus_wins[i]       for i in best],
    })
    if 'Event' in df_full.columns:
        out.insert(1, 'True_Event', df_full['Event'].values)
    out.to_csv(os.path.join(output_dir,'predictions.csv'), index=False)
    print(f'  Row-level   → {output_dir}/predictions.csv  ({len(out):,} rows)')

    pd.DataFrame({'TIMESTAMP':ts_wins,'Predicted_Event':y_pred_wins,
                  'Predicted_Location':y_bus_wins}).to_csv(
        os.path.join(output_dir,'predictions_windows.csv'), index=False)
    print(f'  Window-level→ {output_dir}/predictions_windows.csv  '
          f'({len(ts_wins):,} rows)')


print('=' * 62)
print('PHASE 6 — Saving Predictions & Models')
print('=' * 62)

# Full-dataset prediction
X_all_s = scaler.transform(X).astype(np.float32)
y_full_pred, y_full_bus, _ = predict_all(X_all_s)
save_predictions(df, y_full_pred, y_full_bus, ts, OUTPUT_DIR)

os.makedirs(OUTPUT_DIR, exist_ok=True)
joblib.dump(scaler,     os.path.join(OUTPUT_DIR,'scaler.pkl'))
joblib.dump(detector,   os.path.join(OUTPUT_DIR,'xgb_detector.pkl'))
joblib.dump(classifier, os.path.join(OUTPUT_DIR,'lgb_classifier.pkl'))
joblib.dump(le,         os.path.join(OUTPUT_DIR,'label_encoder.pkl'))
if fallback_localizer is not None:
    joblib.dump(fallback_localizer, os.path.join(OUTPUT_DIR,'lgb_fallback_localizer.pkl'))
joblib.dump({
    'BEST_THRESH':   BEST_THRESH,
    'feature_cols':  feature_cols,
    'WINDOW_SIZES':  WINDOW_SIZES,
    'WINDOW_STRIDE': WINDOW_STRIDE,
    'SAMPLE_RATE':   SAMPLE_RATE,
    'n_params':      n_params,
    'eff_val':       float(eff_val),
    'eff_test':      float(eff_test),
}, os.path.join(OUTPUT_DIR,'config.pkl'))

metrics_out = {k: (float(v) if isinstance(v,(float,np.floating)) else v)
               for k, v in {
                   **{f'val_{k}':v  for k,v in val_metrics.items()  if k!='confusion_matrix'},
                   **{f'test_{k}':v for k,v in test_metrics.items() if k!='confusion_matrix'},
                   'n_params':n_params,'eff_val':eff_val,'eff_test':eff_test
               }.items()}
with open(os.path.join(OUTPUT_DIR,'metrics.json'),'w') as f:
    json.dump(metrics_out, f, indent=2, default=str)

print(f'\n  Models saved to {OUTPUT_DIR}/')
print(f'\n  ══ PIPELINE COMPLETE ══')
print(f'  Val  Macro-F1         : {val_metrics["macro_f1"]:.4f}')
print(f'  Test Macro-F1         : {test_metrics["macro_f1"]:.4f}')
print(f'  Test Detection F1     : {test_metrics["detection_f1"]:.4f}')
print(f'  Test Loc Top-1        : {test_metrics["localization_top1"]:.4f}')
print(f'  Test Loc Top-3        : {test_metrics["localization_top3"]:.4f}')
print(f'  Test Efficiency Score : {eff_test:.4f}')
print(f'  Total Parameters      : {n_params:,}')

PHASE 6 — Saving Predictions & Models
  Row-level   → ./predictions/predictions.csv  (150,641 rows)
  Window-level→ ./predictions/predictions_windows.csv  (10,041 rows)

  Models saved to ./predictions/

  ══ PIPELINE COMPLETE ══
  Val  Macro-F1         : 1.0000
  Test Macro-F1         : 0.6012
  Test Detection F1     : 0.9654
  Test Loc Top-1        : 0.8954
  Test Loc Top-3        : 0.8954
  Test Efficiency Score : 0.4729
  Total Parameters      : 18,958
